# Milestone 9c — GPT-5.6 Sol with the selected context-engineered system

## Goal

Test the highest-leverage remaining hypothesis: **does a newer, more capable model improve Text-to-SQL accuracy when the system context is held fixed?**

This notebook evaluates GPT-5.6 Sol on:

- the visible 12-question `working_dev` set; and
- the fixed 24-question `frozen_eval` benchmark.

The selected Milestone 5 context, prompt format, SQL normalizer, database, evaluator, and one-attempt policy remain unchanged. The primary comparison is GPT-5.2 with engineered context versus GPT-5.6 Sol with that same context.

This is a **model-capability ablation**, not another prompt-tuning round.

## Research question and decision rule

> With the same context-engineered system, does GPT-5.6 Sol materially outperform GPT-5.2?

| Outcome | Interpretation |
|---|---|
| Reaches at least 20/24 | The model upgrade clears the practical 80% target without an agentic workflow |
| Improves but remains below 20/24 | Model capability helps, but a bounded verification workflow may still be justified |
| No improvement | A model upgrade alone does not address the residual error patterns |
| Regression | The newer model or selected reasoning setting is a worse fit for this task |

Because 80% of 24 is 19.2, the smallest whole-number result meeting the target is **20/24 (83.3%)**.

## Controlled experiment contract

| Component | Fixed value |
|---|---|
| Candidate model | GPT-5.6 Sol deployment from `AZURE_AI_ADVANCED_MODEL_DEPLOYMENT_NAME` |
| API | Azure Foundry OpenAI Responses API |
| Reasoning effort | `medium` |
| Instructions and schema context | Exact selected Milestone 5 contract |
| Prompt format | Existing `Database schema` + `Question` format |
| Output normalization | Strict removal of one surrounding SQL fence |
| Attempts | One valid response per question |
| Candidate repair | Disabled |
| Incorrect-answer retry | Disabled |
| Evaluation | Same SQLite comparator and reference-only compatibility rules |

The context file retains `slm` in its filename because it records the provenance of the selected Milestone 5 context. Its text is model-agnostic and is intentionally reused unchanged.

Model generation can vary between calls; the **evaluation procedure and experiment contract** are deterministic. Saved artifacts make the observed run reproducible and auditable.

In [1]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import json
import os
import sqlite3

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

from domain_adapt.context_engineering import (
    load_context_engineering_config,
    request_frontier_context_engineered_sql,
)
from domain_adapt.evaluation import compare_sql_results
from domain_adapt.evaluation_analysis import (
    compare_runs,
    failure_category,
    index_results,
    summarize_results,
)
from domain_adapt.finch import make_reference_compatible


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_crlf_text(path: Path) -> str:
    # Existing config hashes were recorded on Windows. Canonical newline
    # handling keeps validation stable on Windows and Linux.
    canonical = path.read_text(encoding="utf-8").replace("\r\n", "\n")
    canonical = canonical.replace("\r", "\n").replace("\n", "\r\n")
    return sha256(canonical.encode("utf-8")).hexdigest()


def save_json_atomic(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(value, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


REPOSITORY_ROOT = find_repository_root(Path.cwd())
DATA_DIR = REPOSITORY_ROOT / "data"
RUN_DIR = DATA_DIR / "runs"
SEALED_RUN_DIR = RUN_DIR / "sealed"

DB_PATH = DATA_DIR / "financial.sqlite"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
MANIFEST_PATH = REPOSITORY_ROOT / "configs" / "eval_manifest_v1.json"
ADVANCED_CONFIG_PATH = (
    REPOSITORY_ROOT
    / "configs"
    / "gpt_5_6_sol_context_engineered_v1.json"
)
CONTEXT_CONTRACT_PATH = (
    REPOSITORY_ROOT / "configs" / "slm_context_engineered_v1.json"
)

EXPECTED_ADVANCED_CONFIG_SHA256 = (
    "7ba11af0d6c5c75a3428dea58dc44633631f8e4d8435a3627fc039718202c157"
)
EXPECTED_CONTEXT_CONTRACT_SHA256 = (
    "574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68"
)
EXPECTED_INSTRUCTIONS_SHA256 = (
    "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509"
)
EXPECTED_SCHEMA_CONTEXT_SHA256 = (
    "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987"
)

load_dotenv(REPOSITORY_ROOT / ".env", override=True)
print("Repository root:", REPOSITORY_ROOT)

Repository root: C:\Users\shchitt\Downloads\Projects\domain-adapt


## 1. Validate both configuration layers

The advanced-model configuration records the deployment variable, API, reasoning effort, and immutable reference to the selected context contract. The context contract contains the actual instructions and schema text.

Hash validation prevents accidental prompt drift. This cell makes no model request.

In [2]:
for required_path in (
    DB_PATH,
    DATASET_PATH,
    MANIFEST_PATH,
    ADVANCED_CONFIG_PATH,
    CONTEXT_CONTRACT_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

assert sha256_crlf_text(ADVANCED_CONFIG_PATH) == (
    EXPECTED_ADVANCED_CONFIG_SHA256
)
assert sha256_crlf_text(CONTEXT_CONTRACT_PATH) == (
    EXPECTED_CONTEXT_CONTRACT_SHA256
)

advanced_config = json.loads(
    ADVANCED_CONFIG_PATH.read_text(encoding="utf-8")
)
context_contract = load_context_engineering_config(CONTEXT_CONTRACT_PATH)

assert advanced_config["api"] == "responses"
assert advanced_config["deployment_env"] == (
    "AZURE_AI_ADVANCED_MODEL_DEPLOYMENT_NAME"
)
assert advanced_config["project_endpoint_env"] == (
    "AZURE_AI_PROJECT_ENDPOINT"
)
assert advanced_config["context_contract_sha256"] == (
    EXPECTED_CONTEXT_CONTRACT_SHA256
)
assert advanced_config["reasoning_effort"] == "medium"
assert advanced_config["generation"]["attempts_per_question"] == 1
assert advanced_config["generation"]["candidate_sql_repair"] is False
assert advanced_config["generation"]["retry_incorrect_answers"] is False

assert context_contract["instructions_sha256"] == (
    EXPECTED_INSTRUCTIONS_SHA256
)
assert context_contract["schema_context_sha256"] == (
    EXPECTED_SCHEMA_CONTEXT_SHA256
)

print("Advanced-model wrapper contract validated.")
print("Context contract validated.")
print("Reasoning effort:", advanced_config["reasoning_effort"])
print("No model request made.")

Advanced-model wrapper contract validated.
Context contract validated.
Reasoning effort: medium
No model request made.


## 2. Reconstruct the leakage-safe evaluation sets

The 12-question working set is a visible technical and directional check. The fixed 24-question set is the official comparison benchmark. Membership and disjointness are validated before opening the database.

In [3]:
records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)


def select_examples(question_ids: set[int]) -> list[dict]:
    return sorted(
        [
            row for row in records
            if row["db_name"] == "bird"
            and row["db_id"] == "financial"
            and row["question_id"] in question_ids
        ],
        key=lambda row: row["question_id"],
    )


working_examples = select_examples(working_ids)
frozen_examples = select_examples(frozen_ids)

assert len(working_examples) == 12
assert len(frozen_examples) == 24
assert {row["question_id"] for row in working_examples} == working_ids
assert {row["question_id"] for row in frozen_examples} == frozen_ids

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True,
)

print("Working questions:", len(working_examples))
print("Frozen questions:", len(frozen_examples))
print("Overlap:", len(working_ids & frozen_ids))

Working questions: 12
Frozen questions: 24
Overlap: 0


## 3. Create the Azure Foundry project client

GPT-5.6 Sol was deployed in the Foundry project resource, so this notebook uses `AZURE_AI_PROJECT_ENDPOINT` and `AIProjectClient`. `DefaultAzureCredential` can reuse an active Azure CLI login.

If authentication fails, run `az login --tenant <your-tenant-id>` in PowerShell and rerun this cell. Creating the client itself does not call the model.

In [4]:
deployment_name = os.environ[
    advanced_config["deployment_env"]
].strip()
project_endpoint = os.environ[
    advanced_config["project_endpoint_env"]
].strip()

if not deployment_name:
    raise ValueError("Advanced-model deployment name is empty")
if not project_endpoint:
    raise ValueError("Azure AI project endpoint is empty")

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
)
advanced_client = project_client.get_openai_client()

print("Deployment:", deployment_name)
print("Declared model:", advanced_config["declared_underlying_model"])
print("Reasoning effort:", advanced_config["reasoning_effort"])
print("No model request made.")

Deployment: gpt-5.6-sol
Declared model: gpt-5.6-sol
Reasoning effort: medium
No model request made.


## 4. Optional smoke test

Question 16 belongs to the visible working set. Use it once to verify deployment routing, the Responses API, reasoning parameter, strict normalization, SQL execution, and evaluation before starting either batch.

In [5]:
import inspect

print(inspect.getfile(request_frontier_context_engineered_sql))
print(inspect.signature(request_frontier_context_engineered_sql))

C:\Users\shchitt\Downloads\Projects\domain-adapt\src\domain_adapt\context_engineering.py
(*, client: Any, deployment_name: str, question: str, config: dict[str, typing.Any], reasoning_effort: str | None = None) -> tuple[typing.Any, str, str, bool]


In [6]:
RUN_QUESTION_16_SMOKE = True

if RUN_QUESTION_16_SMOKE:
    example = next(row for row in working_examples if row["question_id"] == 16)
    response, raw_sql, normalized_sql, was_unwrapped = (
        request_frontier_context_engineered_sql(
            client=advanced_client,
            deployment_name=deployment_name,
            question=example["question"],
            config=context_contract,
            reasoning_effort=advanced_config["reasoning_effort"],
        )
    )
    reference_sql, rules = make_reference_compatible(example["SQL"])
    comparison = compare_sql_results(
        conn=conn,
        reference_sql=reference_sql,
        candidate_sql=normalized_sql,
    )
    print("Response model:", response.model)
    print("Raw SQL:\n", raw_sql)
    print("Normalized SQL:\n", normalized_sql)
    print("Fence unwrapped:", was_unwrapped)
    print("Passed:", comparison.equivalent)
    print("Reason:", comparison.reason)
    print("Reference-only rules:", rules)
    print("Usage:", response.usage)
else:
    print("Question 16 smoke test skipped.")

Response model: gpt-5.6-sol
Raw SQL:
 SELECT DISTINCT account.date
FROM trans
JOIN account ON trans.account_id = account.account_id
WHERE trans.amount = 840
  AND trans.date = '1998-10-14';
Normalized SQL:
 SELECT DISTINCT account.date
FROM trans
JOIN account ON trans.account_id = account.account_id
WHERE trans.amount = 840
  AND trans.date = '1998-10-14';
Fence unwrapped: False
Passed: True
Reason: unordered results match
Reference-only rules: ['case_insensitive_text_equality']
Usage: ResponseUsage(input_tokens=1265, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=1262), output_tokens=43, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1308)


## 5. Reusable, resumable collector

The same function handles both splits. Each question receives exactly one request, then the candidate is normalized, executed, compared, and atomically saved.

- Working outcomes are visible for diagnosis.
- Frozen collection prints only counts to reduce observer influence.
- API failures are recorded, never silently retried as correctness attempts.

In [7]:
def collect_results(
    *,
    run: dict,
    examples: list[dict],
    output_path: Path,
    reveal_outcomes: bool,
) -> None:
    completed_ids = set(index_results(run["results"]))
    pending_examples = [
        example
        for example in examples
        if example["question_id"] not in completed_ids
    ]

    for example in pending_examples:
        question_id = example["question_id"]
        started = perf_counter()

        try:
            response, raw_sql, normalized_sql, was_unwrapped = (
                request_frontier_context_engineered_sql(
                    client=advanced_client,
                    deployment_name=deployment_name,
                    question=example["question"],
                    config=context_contract,
                    reasoning_effort=advanced_config["reasoning_effort"],
                )
            )
            reference_sql, rules = make_reference_compatible(example["SQL"])
            comparison = compare_sql_results(
                conn=conn,
                reference_sql=reference_sql,
                candidate_sql=normalized_sql,
            )
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": raw_sql,
                "generated_sql": normalized_sql,
                "output_envelope_normalized": was_unwrapped,
                "passed": comparison.equivalent,
                "reason": comparison.reason,
                "expected_row_count": comparison.expected_row_count,
                "candidate_row_count": comparison.candidate_row_count,
                "reference_compatibility_rules": rules,
                "response_id": response.id,
                "response_status": response.status,
                "response_model": response.model,
                "usage": response.usage.model_dump() if response.usage else {},
                "api_error": None,
            }
        except Exception as error:
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": None,
                "generated_sql": None,
                "output_envelope_normalized": False,
                "passed": False,
                "reason": "model request failed",
                "expected_row_count": None,
                "candidate_row_count": None,
                "reference_compatibility_rules": None,
                "response_id": None,
                "response_status": None,
                "response_model": None,
                "usage": {},
                "api_error": f"{type(error).__name__}: {error}",
            }

        result["latency_seconds"] = round(perf_counter() - started, 3)
        run["results"].append(result)
        index_results(run["results"])
        save_json_atomic(output_path, run)

        if reveal_outcomes:
            outcome = "PASS" if result["passed"] else "FAIL"
            print(
                f"[{len(run['results']):02d}/{len(examples):02d}] "
                f"ID {question_id:>3} ({example['difficulty']:<6}) "
                f"{outcome}: {result['reason']}"
            )
        else:
            print(f"Completed {len(run['results'])}/{len(examples)}")

## 6. Run the 12-question working evaluation

Set the flag to `True` for the first batch. The artifact is resumable: rerunning the cell skips completed IDs and rejects contract drift or duplicates.

The working set is diagnostic. Do not change the selected context after viewing these outcomes; doing so would turn the frozen comparison into another prompt-selection cycle.

In [8]:
WORKING_RUN_PATH = (
    RUN_DIR / "gpt_5_6_sol_context_engineered_working_dev_v1.json"
)
RUN_WORKING_EVALUATION = True

base_run_contract = {
    "benchmark": manifest["benchmark"],
    "variant": "gpt-5.6-sol-context-engineered-v1",
    "deployment_name": deployment_name,
    "declared_underlying_model": advanced_config["declared_underlying_model"],
    "reasoning_effort": advanced_config["reasoning_effort"],
    "advanced_config_sha256": EXPECTED_ADVANCED_CONFIG_SHA256,
    "context_contract_sha256": EXPECTED_CONTEXT_CONTRACT_SHA256,
    "instructions_sha256": EXPECTED_INSTRUCTIONS_SHA256,
    "schema_context_sha256": EXPECTED_SCHEMA_CONTEXT_SHA256,
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "retry_incorrect_answers": False,
    "output_normalization": context_contract["output_normalization"],
}

working_contract = {
    **base_run_contract,
    "run_version": 1,
    "split": "working_dev",
}

if WORKING_RUN_PATH.exists():
    working_run = json.loads(WORKING_RUN_PATH.read_text(encoding="utf-8"))
    for key, expected_value in working_contract.items():
        assert working_run[key] == expected_value
    index_results(working_run["results"])
    print("Resuming working results:", len(working_run["results"]))
elif RUN_WORKING_EVALUATION:
    working_run = {
        **working_contract,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": [],
    }
    save_json_atomic(WORKING_RUN_PATH, working_run)
else:
    working_run = None
    print("Working evaluation disabled; no local artifact found.")

if RUN_WORKING_EVALUATION:
    assert working_run is not None
    collect_results(
        run=working_run,
        examples=working_examples,
        output_path=WORKING_RUN_PATH,
        reveal_outcomes=True,
    )
else:
    print("No working model requests made.")

[01/12] ID   5 (medium) PASS: unordered results match
[02/12] ID  11 (medium) PASS: ordered results match
[03/12] ID  14 (easy  ) FAIL: unordered results differ
[04/12] ID  16 (easy  ) PASS: unordered results match
[05/12] ID  45 (easy  ) FAIL: ordered results differ
[06/12] ID  49 (medium) PASS: unordered results match
[07/12] ID  61 (hard  ) FAIL: unordered results differ
[08/12] ID  69 (easy  ) PASS: unordered results match
[09/12] ID  81 (hard  ) FAIL: unordered results differ
[10/12] ID  89 (easy  ) PASS: unordered results match
[11/12] ID  99 (easy  ) PASS: unordered results match
[12/12] ID 105 (medium) FAIL: column count differs: 3 != 2


In [9]:
if WORKING_RUN_PATH.exists():
    working_run = json.loads(WORKING_RUN_PATH.read_text(encoding="utf-8"))
    working_results = working_run["results"]
    assert len(working_results) == 12
    assert set(index_results(working_results)) == working_ids
    working_summary = summarize_results(working_results)
    working_run["summary"] = working_summary
    save_json_atomic(WORKING_RUN_PATH, working_run)

    print("GPT-5.6 Sol working evaluation complete.")
    print(
        f"Overall: {working_summary['passed']}/12 "
        f"({working_summary['accuracy']:.1%})"
    )
    print(
        f"Executable: {working_summary['executable']}/12 "
        f"({working_summary['executable_rate']:.1%})"
    )
    print("By difficulty:", working_summary["by_difficulty"])
    print("Failure categories:", working_summary["failure_categories"])
    print("API errors:", working_summary["api_errors"])
    print("Token usage:", working_summary["token_usage"])
    print("Median latency:", working_summary["median_latency_seconds"])
    print("Artifact SHA-256:", sha256_file(WORKING_RUN_PATH))
else:
    working_summary = None
    print("Working summary unavailable.")

GPT-5.6 Sol working evaluation complete.
Overall: 7/12 (58.3%)
Executable: 12/12 (100.0%)
By difficulty: {'easy': {'passed': 4, 'total': 6, 'accuracy': 0.6666666666666666}, 'medium': {'passed': 3, 'total': 4, 'accuracy': 0.75}, 'hard': {'passed': 0, 'total': 2, 'accuracy': 0.0}}
Failure categories: {'unordered_results_differ': 3, 'ordered_results_differ': 1, 'column_count_differs': 1}
API errors: 0
Token usage: {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 17126}
Median latency: 3.0255
Artifact SHA-256: 35ecfe4fe2c419df558619c3aa71426a0331d906539c39bce971803875e041bf


## 7. Run the fixed 24-question benchmark

Start this only after the smoke test and working batch are technically healthy. The benchmark uses the existing fixed IDs; it is an additional fixed-system comparison, not a newly untouched test.

Set `RUN_FROZEN_EVALUATION = True` once. Progress is resumable and saved after every request.

In [10]:
FROZEN_RUN_PATH = (
    SEALED_RUN_DIR / "gpt_5_6_sol_context_engineered_frozen_eval_v1.json"
)
RUN_FROZEN_EVALUATION = True

frozen_contract = {
    **base_run_contract,
    "run_version": 1,
    "split": "frozen_eval",
}

if FROZEN_RUN_PATH.exists():
    frozen_run = json.loads(FROZEN_RUN_PATH.read_text(encoding="utf-8"))
    for key, expected_value in frozen_contract.items():
        assert frozen_run[key] == expected_value
    index_results(frozen_run["results"])
    print("Resuming frozen results:", len(frozen_run["results"]))
elif RUN_FROZEN_EVALUATION:
    frozen_run = {
        **frozen_contract,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": [],
    }
    save_json_atomic(FROZEN_RUN_PATH, frozen_run)
else:
    frozen_run = None
    print("Frozen evaluation disabled; no local artifact found.")

if RUN_FROZEN_EVALUATION:
    assert frozen_run is not None
    collect_results(
        run=frozen_run,
        examples=frozen_examples,
        output_path=FROZEN_RUN_PATH,
        reveal_outcomes=False,
    )
else:
    print("No frozen model requests made.")

Completed 1/24
Completed 2/24
Completed 3/24
Completed 4/24
Completed 5/24
Completed 6/24
Completed 7/24
Completed 8/24
Completed 9/24
Completed 10/24
Completed 11/24
Completed 12/24
Completed 13/24
Completed 14/24
Completed 15/24
Completed 16/24
Completed 17/24
Completed 18/24
Completed 19/24
Completed 20/24
Completed 21/24
Completed 22/24
Completed 23/24
Completed 24/24


## 8. Infrastructure-only completion

Correctness retries remain forbidden. If authentication or infrastructure failed before a response was produced, those missing requests may be completed once while preserving every valid response.

Enable this only when the diagnostic reports infrastructure-missing responses.

In [11]:
COMPLETED_FROZEN_RUN_PATH = (
    SEALED_RUN_DIR / "gpt_5_6_sol_context_engineered_frozen_eval_v2.json"
)
COMPLETE_INFRASTRUCTURE_FAILURES = False


def is_infrastructure_only_failure(result: dict) -> bool:
    return (
        result.get("api_error") is not None
        and result.get("response_id") is None
        and result.get("generated_sql") is None
        and not result.get("usage")
    )


if FROZEN_RUN_PATH.exists():
    initial_frozen_run = json.loads(
        FROZEN_RUN_PATH.read_text(encoding="utf-8")
    )
    infrastructure_failures = [
        result
        for result in initial_frozen_run["results"]
        if is_infrastructure_only_failure(result)
    ]
    valid_frozen_results = [
        result
        for result in initial_frozen_run["results"]
        if not is_infrastructure_only_failure(result)
    ]
    print("Valid responses:", len(valid_frozen_results))
    print("Infrastructure-missing responses:", len(infrastructure_failures))
else:
    initial_frozen_run = None
    infrastructure_failures = []
    valid_frozen_results = []
    print("Initial frozen artifact unavailable.")

Valid responses: 24
Infrastructure-missing responses: 0


In [12]:
if COMPLETE_INFRASTRUCTURE_FAILURES:
    assert initial_frozen_run is not None
    assert infrastructure_failures

    if COMPLETED_FROZEN_RUN_PATH.exists():
        completed_frozen_run = json.loads(
            COMPLETED_FROZEN_RUN_PATH.read_text(encoding="utf-8")
        )
        assert completed_frozen_run["source_run_sha256"] == sha256_file(
            FROZEN_RUN_PATH
        )
        index_results(completed_frozen_run["results"])
    else:
        completed_frozen_run = {
            **{
                key: value
                for key, value in initial_frozen_run.items()
                if key not in {"results", "summary"}
            },
            "run_version": 2,
            "source_run": FROZEN_RUN_PATH.name,
            "source_run_sha256": sha256_file(FROZEN_RUN_PATH),
            "completion_policy": (
                "Repeat only requests that received no model response "
                "because of an infrastructure failure."
            ),
            "results": valid_frozen_results,
        }
        save_json_atomic(COMPLETED_FROZEN_RUN_PATH, completed_frozen_run)

    missing_ids = {
        result["question_id"] for result in infrastructure_failures
    }
    completed_ids = set(index_results(completed_frozen_run["results"]))
    completion_examples = [
        example
        for example in frozen_examples
        if example["question_id"] in missing_ids
        and example["question_id"] not in completed_ids
    ]
    collect_results(
        run=completed_frozen_run,
        examples=completion_examples,
        output_path=COMPLETED_FROZEN_RUN_PATH,
        reveal_outcomes=False,
    )
else:
    print("Infrastructure completion skipped.")

Infrastructure completion skipped.


## 9. Select and summarize the official GPT-5.6 Sol run

The v2 artifact is selected only if infrastructure completion was needed. An official result requires all 24 unique IDs and zero API errors.

In [13]:
official_frozen_path = None
official_frozen_run = None

for candidate_path in (COMPLETED_FROZEN_RUN_PATH, FROZEN_RUN_PATH):
    if not candidate_path.exists():
        continue
    candidate_run = json.loads(candidate_path.read_text(encoding="utf-8"))
    candidate_results = candidate_run["results"]
    if (
        len(candidate_results) == 24
        and set(index_results(candidate_results)) == frozen_ids
        and all(result.get("api_error") is None for result in candidate_results)
    ):
        official_frozen_path = candidate_path
        official_frozen_run = candidate_run
        break

if official_frozen_run is None:
    frozen_summary = None
    official_frozen_sha256 = None
    print("Official GPT-5.6 Sol frozen run is not complete.")
else:
    frozen_summary = summarize_results(official_frozen_run["results"])
    official_frozen_run["summary"] = frozen_summary
    save_json_atomic(official_frozen_path, official_frozen_run)
    official_frozen_sha256 = sha256_file(official_frozen_path)

    print("Official GPT-5.6 Sol evaluation complete.")
    print(
        f"Overall: {frozen_summary['passed']}/24 "
        f"({frozen_summary['accuracy']:.1%})"
    )
    print(
        f"Executable: {frozen_summary['executable']}/24 "
        f"({frozen_summary['executable_rate']:.1%})"
    )
    print("By difficulty:", frozen_summary["by_difficulty"])
    print("Failure categories:", frozen_summary["failure_categories"])
    print("API errors:", frozen_summary["api_errors"])
    print("Token usage:", frozen_summary["token_usage"])
    print("Median latency:", frozen_summary["median_latency_seconds"])
    print("Sealed-run SHA-256:", official_frozen_sha256)

Official GPT-5.6 Sol evaluation complete.
Overall: 17/24 (70.8%)
Executable: 24/24 (100.0%)
By difficulty: {'easy': {'passed': 10, 'total': 12, 'accuracy': 0.8333333333333334}, 'medium': {'passed': 5, 'total': 9, 'accuracy': 0.5555555555555556}, 'hard': {'passed': 2, 'total': 3, 'accuracy': 0.6666666666666666}}
Failure categories: {'ordered_results_differ': 2, 'column_count_differs': 1, 'unordered_results_differ': 4}
API errors: 0
Token usage: {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 36316}
Median latency: 3.1109999999999998
Sealed-run SHA-256: 6c46d03c90d32c36db8c5c433cd37bfb364c2e3cf6c4bc149980aa88dff7e5d5


## 10. Compare GPT-5.6 Sol directly with GPT-5.2

This is the primary ablation. Both models receive the same engineered context and are evaluated with the same execution comparator. The model deployment is the intended capability change; GPT-5.6 Sol's fixed `medium` reasoning setting is recorded as part of that candidate system.

In [14]:
GPT_5_2_WORKING_PATH = (
    RUN_DIR / "gpt_5_2_context_engineered_working_dev_v1.json"
)
GPT_5_2_FROZEN_PATHS = (
    SEALED_RUN_DIR / "gpt_5_2_context_engineered_frozen_eval_v2.json",
    SEALED_RUN_DIR / "gpt_5_2_context_engineered_frozen_eval_v1.json",
)


def select_complete_artifact(
    paths: tuple[Path, ...],
    expected_ids: set[int],
) -> Path | None:
    for path in paths:
        if not path.exists():
            continue
        run = json.loads(path.read_text(encoding="utf-8"))
        results = run.get("results", [])
        if (
            len(results) == len(expected_ids)
            and set(index_results(results)) == expected_ids
            and all(result.get("api_error") is None for result in results)
        ):
            return path
    return None


def compare_pair(
    label: str,
    baseline_path: Path | None,
    candidate_path: Path | None,
) -> dict | None:
    if (
        baseline_path is None
        or candidate_path is None
        or not baseline_path.exists()
        or not candidate_path.exists()
    ):
        print(f"{label}: comparison artifacts unavailable")
        return None

    baseline_run = json.loads(baseline_path.read_text(encoding="utf-8"))
    candidate_run = json.loads(candidate_path.read_text(encoding="utf-8"))
    baseline_summary = summarize_results(baseline_run["results"])
    candidate_summary = summarize_results(candidate_run["results"])
    transitions = compare_runs(
        baseline_run["results"],
        candidate_run["results"],
    )
    transition_counts = Counter(row["transition"] for row in transitions)

    comparison = {
        "baseline_summary": baseline_summary,
        "candidate_summary": candidate_summary,
        "accuracy_delta": (
            candidate_summary["accuracy"] - baseline_summary["accuracy"]
        ),
        "executable_rate_delta": (
            candidate_summary["executable_rate"]
            - baseline_summary["executable_rate"]
        ),
        "total_token_delta": (
            candidate_summary["token_usage"]["total_tokens"]
            - baseline_summary["token_usage"]["total_tokens"]
        ),
        "median_latency_delta": (
            candidate_summary["median_latency_seconds"]
            - baseline_summary["median_latency_seconds"]
        ),
        "transition_counts": dict(transition_counts),
    }

    print(f"{label}:")
    print(
        "  Accuracy:",
        f"{baseline_summary['passed']}/{baseline_summary['total']} -> "
        f"{candidate_summary['passed']}/{candidate_summary['total']}",
    )
    print(
        "  Executable:",
        f"{baseline_summary['executable']}/{baseline_summary['total']} -> "
        f"{candidate_summary['executable']}/{candidate_summary['total']}",
    )
    print("  Accuracy delta:", f"{comparison['accuracy_delta']:+.1%}")
    print(
        "  Executable delta:",
        f"{comparison['executable_rate_delta']:+.1%}",
    )
    print("  Token delta:", f"{comparison['total_token_delta']:+,}")
    print("  Latency delta:", f"{comparison['median_latency_delta']:+.3f}s")
    print("  Outcome transitions:", comparison["transition_counts"])
    return comparison


official_gpt_5_2_frozen_path = select_complete_artifact(
    GPT_5_2_FROZEN_PATHS,
    frozen_ids,
)

working_comparison = compare_pair(
    "Working set: GPT-5.2 -> GPT-5.6 Sol",
    GPT_5_2_WORKING_PATH,
    WORKING_RUN_PATH,
)
frozen_comparison = compare_pair(
    "Frozen benchmark: GPT-5.2 -> GPT-5.6 Sol",
    official_gpt_5_2_frozen_path,
    official_frozen_path,
)

Working set: GPT-5.2 -> GPT-5.6 Sol:
  Accuracy: 6/12 -> 7/12
  Executable: 12/12 -> 12/12
  Accuracy delta: +8.3%
  Executable delta: +0.0%
  Token delta: +1,227
  Latency delta: +1.510s
  Outcome transitions: {'still_passes': 6, 'still_fails': 5, 'improved': 1}
Frozen benchmark: GPT-5.2 -> GPT-5.6 Sol:
  Accuracy: 16/24 -> 17/24
  Executable: 24/24 -> 24/24
  Accuracy delta: +4.2%
  Executable delta: +0.0%
  Token delta: +4,199
  Latency delta: +1.588s
  Outcome transitions: {'still_passes': 16, 'still_fails': 7, 'improved': 1}


## 11. Updated benchmark matrix

This table places the advanced-model result beside the earlier capability, context, and SFT experiments. Token totals measure traffic, not monetary cost; model prices can differ.

In [15]:
official_sft_path = select_complete_artifact(
    (
        SEALED_RUN_DIR / "ministral_3b_sft_v1_frozen_eval_v2.json",
        SEALED_RUN_DIR / "ministral_3b_sft_v1_frozen_eval_v1.json",
    ),
    frozen_ids,
)

FROZEN_VARIANT_PATHS = {
    "GPT-5.2 + minimal context": (
        SEALED_RUN_DIR / "gpt_5_2_minimal_frozen_eval_v1.json"
    ),
    "GPT-5.2 + engineered context": official_gpt_5_2_frozen_path,
    "GPT-5.6 Sol + engineered context": official_frozen_path,
    "Ministral-3B + minimal context": (
        SEALED_RUN_DIR / "ministral_3b_minimal_frozen_eval_v1.json"
    ),
    "Ministral-3B + engineered context": (
        SEALED_RUN_DIR
        / "ministral_3b_context_engineered_frozen_eval_v2.json"
    ),
    "SFT v1 + engineered context": official_sft_path,
}

variant_summaries = {}
for name, path in FROZEN_VARIANT_PATHS.items():
    if path is None or not path.exists():
        continue
    run = json.loads(path.read_text(encoding="utf-8"))
    variant_summaries[name] = summarize_results(run["results"])

print(
    f"{'Variant':<42} {'Accuracy':>10} "
    f"{'Executable':>12} {'Tokens':>10} {'Latency':>9}"
)
print("-" * 89)
for name, summary in variant_summaries.items():
    latency = summary["median_latency_seconds"]
    latency_text = "n/a" if latency is None else f"{latency:.3f}s"
    print(
        f"{name:<42} "
        f"{summary['passed']}/{summary['total']:<7} "
        f"{summary['executable']}/{summary['total']:<9} "
        f"{summary['token_usage']['total_tokens']:>10,} "
        f"{latency_text:>9}"
    )

Variant                                      Accuracy   Executable     Tokens   Latency
-----------------------------------------------------------------------------------------
GPT-5.2 + minimal context                  11/24      24/24             9,081    1.827s
GPT-5.2 + engineered context               16/24      24/24            32,117    1.523s
GPT-5.6 Sol + engineered context           17/24      24/24            36,316    3.111s
Ministral-3B + minimal context             1/24      10/24             8,254    0.940s
Ministral-3B + engineered context          7/24      17/24            34,241    1.079s
SFT v1 + engineered context                7/24      16/24            34,306    1.237s


## 12. Inspect residual failure movement

Aggregate accuracy alone can hide churn. This deterministic analysis reports whether the advanced model fixed or introduced individual failures and how the residual categories moved.

These observations inform the next architecture decision; they must not be used to retune this already-evaluated configuration.

In [16]:
if (
    official_frozen_run is not None
    and official_gpt_5_2_frozen_path is not None
):
    gpt_5_2_run = json.loads(
        official_gpt_5_2_frozen_path.read_text(encoding="utf-8")
    )
    transitions = compare_runs(
        gpt_5_2_run["results"],
        official_frozen_run["results"],
    )
    transition_counts = Counter(row["transition"] for row in transitions)
    print("Outcome transitions:", dict(transition_counts))

    gpt_5_2_by_id = index_results(gpt_5_2_run["results"])
    advanced_by_id = index_results(official_frozen_run["results"])
    residual_transitions = Counter()
    for question_id in sorted(frozen_ids):
        baseline = gpt_5_2_by_id[question_id]
        advanced = advanced_by_id[question_id]
        if baseline["passed"] or advanced["passed"]:
            continue
        residual_transitions[
            (failure_category(baseline), failure_category(advanced))
        ] += 1

    print("Residual failure-category transitions:")
    for transition, count in residual_transitions.most_common():
        print(f"- {transition[0]} -> {transition[1]}: {count}")
else:
    transition_counts = None
    residual_transitions = None
    print("Failure-transition analysis is not ready.")

Outcome transitions: {'still_passes': 16, 'still_fails': 7, 'improved': 1}
Residual failure-category transitions:
- unordered_results_differ -> unordered_results_differ: 4
- ordered_results_differ -> ordered_results_differ: 2
- ordered_results_differ -> column_count_differs: 1


## 13. Record the decision

The result determines the next step:

1. **At least 20/24:** prefer the simpler one-shot advanced-model system; an agentic workflow is optional rather than necessary.
2. **Improvement below 20/24:** model capability is useful, but test a bounded generate → validate → repair workflow on the working set before another frozen run.
3. **No improvement or regression:** do not pay for a model upgrade solely for this task; focus on deterministic validation, semantic checks, and targeted repair.

In [17]:
DECISION_PATH = RUN_DIR / "advanced_model_comparison_v1.json"
TARGET_PASSES = 20

if frozen_comparison is not None and frozen_summary is not None:
    accuracy_delta = frozen_comparison["accuracy_delta"]
    target_reached = frozen_summary["passed"] >= TARGET_PASSES

    if target_reached:
        conclusion = (
            "GPT-5.6 Sol reached the practical accuracy target with the "
            "existing one-shot context-engineered workflow."
        )
        next_step = (
            "Prefer the simpler one-shot advanced-model system and assess "
            "cost before adding workflow complexity."
        )
    elif accuracy_delta > 0:
        conclusion = (
            "GPT-5.6 Sol improved accuracy but did not reach the practical "
            "target. Model capability helps but is not sufficient alone."
        )
        next_step = (
            "Prototype a bounded deterministic validation and one-repair "
            "workflow on working_dev before any additional frozen run."
        )
    elif accuracy_delta == 0:
        conclusion = (
            "GPT-5.6 Sol did not improve frozen accuracy over GPT-5.2 under "
            "the same engineered context."
        )
        next_step = (
            "Prioritize deterministic validation and targeted repair rather "
            "than another model-only upgrade."
        )
    else:
        conclusion = (
            "GPT-5.6 Sol regressed under the selected one-shot contract."
        )
        next_step = (
            "Retain GPT-5.2 as the frontier baseline and test bounded "
            "workflow improvements on working_dev."
        )

    decision_record = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment": "gpt-5.6-sol-model-capability-ablation-v1",
        "reasoning_effort": advanced_config["reasoning_effort"],
        "target_passes": TARGET_PASSES,
        "working_comparison": working_comparison,
        "frozen_comparison": frozen_comparison,
        "target_reached": target_reached,
        "official_frozen_artifact": official_frozen_path.name,
        "official_frozen_sha256": official_frozen_sha256,
        "conclusion": conclusion,
        "next_step": next_step,
    }
    save_json_atomic(DECISION_PATH, decision_record)

    print("Target reached:", target_reached)
    print("Conclusion:", conclusion)
    print("Next step:", next_step)
    print("Decision artifact:", DECISION_PATH)
else:
    print("Decision deferred until both fixed benchmark runs are available.")

Target reached: False
Conclusion: GPT-5.6 Sol improved accuracy but did not reach the practical target. Model capability helps but is not sufficient alone.
Next step: Prototype a bounded deterministic validation and one-repair workflow on working_dev before any additional frozen run.
Decision artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\advanced_model_comparison_v1.json


## 14. Handoff

After the notebook finishes, retain:

- working and frozen accuracy;
- executability, token traffic, and median latency;
- GPT-5.2 → GPT-5.6 Sol outcome transitions;
- whether the 20/24 target was reached; and
- the generated decision artifact.

Do not tune the context from the frozen failures. If the target is missed, the next experiment should be a **bounded, predeclared workflow**—not an open-ended LangGraph agent. LangGraph may be used as orchestration later, but it is not itself the source of accuracy.